### TODO: Miles to km (?)

### TODO: Provide a detailed description of the trip dataset such that there are no pending questions.

### TODO: Evaluate Aggregation Logic especially in regards to Spatial Analysis and Prediction tasks

### TODO: Make Markdown Text look good

### TODO: Add column description from website here as Markdown

### TODO: Add Outlier Analysis

In [1]:
import pandas as pd
import numpy as np
import h3

## What happened before uploading the CSV:
- Filtering the data for:
    - Pickup/Dropoff Census Tract is not null
    - Trip Seconds/Miles is not 0
    - Removing unneccasssary columns: Fare, Tips, Tolls, Extras, Payment Type, Pickup/Dropoff Centroid Location
- resulting data with 16 columns and 6.041.177 rows

In [2]:
taxi_data = pd.read_csv("../data/Taxi_Trips.csv")
taxi_data

,Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Pickup Community Area,Dropoff Community Area,Trip Total,Company,Pickup Centroid Latitude,Pickup Centroid Longitude,Dropoff Centroid Latitude,Dropoff Centroid Longitude
0,a1e30fb944a6fd2598a9c3caf0192121f0e39dea,249d932165de24c10140f979fa81b350183cff7ac63099...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,261.000,"0,4",17031833000,17031833000,28.0,28.0,"$5,00",Globe Taxi,"41,88528132","-87,6572332","41,88528132","-87,6572332"
1,07df4fb41ec699a5287f8a4e64726563bae385fc,fed019ef14a3bfeaa5ef5f523be5d8dc28943952aa66a3...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,240.000,"0,9",17031833000,17031081800,28.0,8.0,"$5,25",Choice Taxi Association Inc,"41,88528132","-87,6572332","41,89321636","-87,63784421"
2,5d9e23fa8a4beb9d495f5c4c7c00bf33026b561e,fb1ed566274f6a66dfcddbe4c014bbe87ac3516412dfe6...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,425.000,"1,41",17031833000,17031320400,28.0,32.0,"$8,45",Sun Taxi,"41,88528132","-87,6572332","41,877406123","-87,621971652"
3,e6ba74242fb88764fb6cb5e34a7f399b613c1a6b,3278a0dd17ed3de47adf8cf7236bf8f21f29e62c04394b...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,1.560,"16,8",17031980000,17031320400,76.0,32.0,"$53,75",Transit Administrative Center Inc,"41,97907082","-87,903039661","41,877406123","-87,621971652"
4,876f1c50d59ac611ff5daca49934a046c951de40,3f365e125eb1d15aae0ae8a4bf53a01191478f26ccf00c...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,1.297,"17,1",17031980000,17031833000,76.0,28.0,"$47,25",5 Star Taxi,"41,97907082","-87,903039661","41,88528132","-87,6572332"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6041172,a75082c74bf8ae2f97d9811101952ba5de8192aa,50fcee6711df1d794e4f337c99f44abe8109795ec69474...,01/01/2024 12:00:00 AM,01/01/2024 12:00:00 AM,452.000,"0,2",17031081700,17031081700,8.0,8.0,"$5,75",Medallion Leasin,"41,892042136","-87,63186395","41,892042136","-87,63186395"
6041173,63d8c865c01bde9e17e469db6a30e33c8cfe5314,259d38cfdbc9ac6f9bb01f0df740e0ddf4a631a70bbdd6...,01/01/2024 12:00:00 AM,01/01/2024 12:00:00 AM,180.000,"0,3",17031081500,17031081201,8.0,8.0,"$5,25","Taxicab Insurance Agency, LLC","41,892507781","-87,626214906","41,899155613","-87,626210532"
6041174,3c05ccf0732fc338b7c875f9a9779039eaada274,0cbf5c0f6aca3628d77c7b6fe89715757ed402a70b0f8b...,01/01/2024 12:00:00 AM,01/01/2024 12:30:00 AM,1.681,"15,34",17031980000,17031071400,76.0,7.0,"$53,10",Globe Taxi,"41,97907082","-87,903039661","41,922082541","-87,634156093"
6041175,ddcd4d6b7c138bee6841a7800cfbb45f31e6101a,0fdab9be71f6d88e3d3a2e115afc5a33d2bf74153792c5...,01/01/2024 12:00:00 AM,01/01/2024 12:45:00 AM,3.059,"17,44",17031980000,17031320100,76.0,32.0,"$66,30",City Service,"41,97907082","-87,903039661","41,884987192","-87,620992913"


In [3]:
taxi_data.info()
taxi_data.describe()

<class 'pandas.DataFrame'>
RangeIndex: 6041177 entries, 0 to 6041176
Data columns (total 16 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Trip ID                     str    
 1   Taxi ID                     str    
 2   Trip Start Timestamp        str    
 3   Trip End Timestamp          str    
 4   Trip Seconds                float64
 5   Trip Miles                  str    
 6   Pickup Census Tract         int64  
 7   Dropoff Census Tract        int64  
 8   Pickup Community Area       float64
 9   Dropoff Community Area      float64
 10  Trip Total                  str    
 11  Company                     str    
 12  Pickup Centroid Latitude    str    
 13  Pickup Centroid Longitude   str    
 14  Dropoff Centroid Latitude   str    
 15  Dropoff Centroid Longitude  str    
dtypes: float64(3), int64(2), str(11)
memory usage: 2.3 GB


,Trip Seconds,Pickup Census Tract,Dropoff Census Tract,Pickup Community Area,Dropoff Community Area
count,6.041177e+06,6.041177e+06,6.041177e+06,6.038883e+06,5.954058e+06
mean,3.380790e+02,1.703147e+10,1.703139e+10,3.371160e+01,2.469550e+01
std,2.973048e+02,3.669296e+05,3.339332e+05,2.415009e+01,1.746181e+01
min,1.000000e+00,1.703101e+10,1.703101e+10,1.000000e+00,1.000000e+00
25%,2.987000e+00,1.703108e+10,1.703108e+10,8.000000e+00,8.000000e+00
50%,3.490000e+02,1.703132e+10,1.703132e+10,3.200000e+01,2.800000e+01
75%,5.690000e+02,1.703184e+10,1.703184e+10,3.300000e+01,3.200000e+01
max,9.990000e+02,1.703198e+10,1.703198e+10,7.700000e+01,7.700000e+01


## 1. Change Data types from string to numeric

In [4]:
# Columns to fix data type
cols_to_fix = [
    'Pickup Centroid Longitude', 'Pickup Centroid Latitude',
    'Dropoff Centroid Longitude', 'Dropoff Centroid Latitude',
    'Trip Total', 'Trip Miles'
]

for col in cols_to_fix:
    
    if col == 'Trip Total':
        # Remove the dollar sign and commas from the Trip Total column
        taxi_data[col] = taxi_data[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
    elif col == 'Trip Miles' or col in ['Pickup Centroid Longitude', 'Pickup Centroid Latitude', 'Dropoff Centroid Longitude', 'Dropoff Centroid Latitude']:
        # Replace the comma with a standard decimal point
        taxi_data[col] = taxi_data[col].astype(str).str.replace(',', '.', regex=False)

    # Convert to numeric
    taxi_data[col] = pd.to_numeric(taxi_data[col], errors='coerce')

# 2. Null Value Analysis

In [5]:
# Find Null Values and add percentage of NaN values for each column
is_na_df = taxi_data.isna().sum()
is_na_df = pd.DataFrame(is_na_df, columns=['NaN Count'])
is_na_df['Total Count'] = len(taxi_data)
is_na_df['NaN Percentage'] = (is_na_df['NaN Count'] / is_na_df['Total Count']) * 100

# Check for any NaN values in all columns
print("NaN values before conversion:")
print(is_na_df)

NaN values before conversion:
                            NaN Count  Total Count  NaN Percentage
Trip ID                             0      6041177        0.000000
Taxi ID                             3      6041177        0.000050
Trip Start Timestamp                0      6041177        0.000000
Trip End Timestamp                  0      6041177        0.000000
Trip Seconds                        0      6041177        0.000000
Trip Miles                          3      6041177        0.000050
Pickup Census Tract                 0      6041177        0.000000
Dropoff Census Tract                0      6041177        0.000000
Pickup Community Area            2294      6041177        0.037973
Dropoff Community Area          87119      6041177        1.442087
Trip Total                      14575      6041177        0.241261
Company                             0      6041177        0.000000
Pickup Centroid Latitude          232      6041177        0.003840
Pickup Centroid Longitude       

## 3. Add Column with H3 index

In [6]:
# Generate H3 indices for pickup and dropoff locations, we use resolution 8 as a starting point
taxi_data['h3_index_pickup'] = [
    # Check if lat and lng are not NaN before converting to H3 index, otherwise return None
    h3.latlng_to_cell(lat, lng, 8) if (lat == lat and lng == lng) else None
    for lat, lng in zip(
        taxi_data['Pickup Centroid Latitude'], 
        taxi_data['Pickup Centroid Longitude']
    )
]
taxi_data['h3_index_dropoff'] = [
    # Check if lat and lng are not NaN before converting to H3 index, otherwise return None
    h3.latlng_to_cell(lat, lng, 8) if (lat == lat and lng == lng) else None
    for lat, lng in zip(
        taxi_data['Dropoff Centroid Latitude'], 
        taxi_data['Dropoff Centroid Longitude']
    )
]

# Check for any NaN values in the new H3 index columns
# TODO: Add percentage of NaN values in the H3 index columns
print(taxi_data[['h3_index_pickup', 
                 'h3_index_dropoff',
                 'Pickup Census Tract', 
                 'Dropoff Census Tract', 
                 'Pickup Community Area', 
                 'Dropoff Community Area']].isna().sum())

columns_to_drop = [
    #'Pickup Centroid Longitude', 'Pickup Centroid Latitude',
    'Dropoff Centroid Longitude', 'Dropoff Centroid Latitude',
    # 'Pickup Census Tract', 'Dropoff Census Tract', # Maybe use this for spatial analysis
    #'Pickup Community Area', 'Dropoff Community Area',
]

# Drop rows where either pickup or dropoff H3 index is NaN, as these rows cannot be used for spatial analysis 
# and drop columns that are not needed for the analysis
# TODO: Consider more sophisticated methods for handling missing values in the coordinate columns, e.g. by imputing with mean/median or using a separate category for missing values, instead of dropping rows with NaN values in the H3 index columns.
taxi_data_processed = taxi_data.drop(columns=columns_to_drop).dropna(subset=['h3_index_pickup', 'h3_index_dropoff', 'Taxi ID', 'Trip Total', 'Trip Miles'])

h3_index_pickup             232
h3_index_dropoff          14311
Pickup Census Tract           0
Dropoff Census Tract          0
Pickup Community Area      2294
Dropoff Community Area    87119
dtype: int64


In [7]:
taxi_data_processed.to_parquet(
    "../data/taxi_data_processed.parquet"
)